In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.express as px
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import itertools

# Leyendo y procesando datos

In [2]:
df= pd.read_csv("Data/spotify-tracks-dataset.csv")
df.drop(columns= ['Unnamed: 0.1', 'Unnamed: 0'], inplace= True)

# Calculando duration en min
df['duration']= df.duration_ms /60_000
df.drop(columns= ['duration_ms'], inplace= True)

# Quitando filas con nan (solo hay una)
df = df.dropna().copy()

# Quitando filas duplicadas
df.drop_duplicates(inplace= True)

#genres_to_remove= ['chill', 'children', 'kids' ,'comedy',
#                   'disney', 'sleep', 'study', 'show-tunes']

#df= df[~df.track_genre.isin(genres_to_remove)]

In [3]:
# Eliminando las siguientes columnas
columns_to_drop = [
    "track_name",
    "album_name",
    "key"
]

df.drop(columns=columns_to_drop, inplace= True)


## Feature engineering (Artistas)

Las canciones pueden tener más de un artista. Por ejemplo, el valor de "artists" la tercera columna es "Ingrid Michaelson;ZAYN".

In [4]:
df.artists.head()

0               Gen Hoshino
1              Ben Woodward
2    Ingrid Michaelson;ZAYN
3              Kina Grannis
4          Chord Overstreet
Name: artists, dtype: object

In [5]:
# Creando una columna auxiliarn "artist_list" con los artistas como lista
df["artist_list"] = (df["artists"].str.split(";").apply(
    lambda x: [a.strip() for a in x]))
# Calculando el número de artistas involucrados en cada canción
df["num_artists"] = df["artist_list"].apply(len)

In [6]:
df.drop_duplicates(subset= "track_id")

,track_id,artists,popularity,explicit,danceability,energy,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration,artist_list,num_artists
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,73,False,0.676,0.4610,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic,3.844433,[Gen Hoshino],1
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,55,False,0.420,0.1660,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic,2.493500,[Ben Woodward],1
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,57,False,0.438,0.3590,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic,3.513767,"[Ingrid Michaelson, ZAYN]",2
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,71,False,0.266,0.0596,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic,3.365550,[Kina Grannis],1
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,82,False,0.618,0.4430,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic,3.314217,[Chord Overstreet],1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,21,False,0.172,0.2350,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music,6.416650,[Rainy Lullaby],1
113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,22,False,0.174,0.1170,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music,6.416667,[Rainy Lullaby],1
113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,22,False,0.629,0.3290,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music,4.524433,[Cesária Evora],1
113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,41,False,0.587,0.5060,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music,4.731550,[Michael W. Smith],1


In [7]:
# Se crea dataframe auxiliar que contiene la popularidad de cada cancion y el nombre de cada artista involucrado
# Si una canción tiene multiples artistas, se crea una fila por cada artista

artist_df = (
    #Quitando canciones duplicadas y usando ["artist_list", "popularity"]
    df.drop_duplicates(subset= "track_id")[["artist_list", "popularity"]]
    #La columna artist_list tiene listas con los artistas involucrados, con explode se crea una fila por cada artista
    .explode("artist_list")
    # Renombrando columna
    .rename(columns={"artist_list": "artist"})
)

In [8]:
# Calculando numero de canciones y popularidad promedio de artista
artist_stats = artist_df.groupby("artist").agg( # Agrupando por artista y calculando
    artist_song_count=("popularity", "size"), # Número de canciones
    artist_mean_popularity=("popularity", "mean") # Popularidad promedio
).reset_index()

# Si el artista tiene menos de "MIN_ARTIST_SONGS" canciones, definimos a artist_mean_popularity como nan
MIN_ARTIST_SONGS = 20

artist_stats.loc[
    artist_stats["artist_song_count"] < MIN_ARTIST_SONGS,
    "artist_mean_popularity"
] = np.nan

artist_stats.sort_values('artist_song_count', ascending= False)

,artist,artist_song_count,artist_mean_popularity
9598,George Jones,332,16.072289
20868,Pritam,323,53.523220
28349,Wolfgang Amadeus Mozart,305,10.416393
1931,Arijit Singh,259,58.166023
10374,Hank Williams,243,17.098765
...,...,...,...
29855,黄霄雲,1,NaN
29856,齋藤摩羅衛門,1,NaN
14819,Leo Abrahams,1,NaN
14820,Leo Anderson,1,NaN


In [9]:
print( f"Artistas con más de {MIN_ARTIST_SONGS} canciones:", artist_stats["artist_mean_popularity"].notna().sum())

Artistas con más de 20 canciones: 1033


In [10]:
# Se crea diccionario con la popularidad promedio de cada artista
artist_pop_lookup = dict(zip( artist_stats["artist"], artist_stats["artist_mean_popularity"]))


In [11]:
# Esta función recibe una lista con artistas y regresa una lista con la popularidad promedio
# de cada artista usando el dic artist_pop_lookup. La función ignora los valores de porpularidad nan


def get_artist_pop_values(artists):
    values = [ artist_pop_lookup[a] for a in artists ]

    values = [v for v in values if pd.notna(v)]

    return values

# Se crea Series con lista de popularidades de los artistas de cada canción
artist_pop_values = df["artist_list"].apply(get_artist_pop_values)

#df["artist_pop_mean"] = artist_pop_values.apply(
#    lambda x: np.mean(x) if len(x) > 0 else np.nan
#)

# Se crea una columna con la popularidad del artista más popular de cada canción. Si la lista no tiene valoeres,
# El valor es nan
df["artist_pop_max"] = artist_pop_values.apply(
    lambda x: np.max(x) if len(x) > 0 else np.nan
)


In [12]:
# df.drop(columns=[ "artists", "artist_list" ], inplace= True) # Quitando columnas "artists" y "artist_list"

In [13]:
# Los artistas con menos de "MIN_ARTIST_SONGS" canciones tiene "artist_pop_max"= nan

df.artist_pop_max.isna().mean()

np.float64(0.5684153977577961)

## Feature engineering (Genre)

En esta sección se calculan features relacionadas con el género de cada canción:
- Popularidad promedio de cada género
- Valores promedios de propiedades de audio de cada género

In [14]:
# Se calcula el valor promedio por género de las siguientes propiedades acústicas de las canciones 
audio_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration",
]

# Calculando popularidad promedio por género
genre_popularity = df.groupby("track_genre")["popularity"].mean().rename("genre_mean_popularity")

# Agregando columna calculada a df
df = df.join( genre_popularity, on="track_genre")

# Calculando valores promedio de columnas en audio_cols
genre_audio_profile = df.groupby("track_genre")[audio_cols].mean().add_prefix("genre_avg_")

# Agregando columnas calculadas al df
df = df.join(genre_audio_profile, on="track_genre")

#Se itera sobre las columnas audio_cols y se calcula la diferencia entre 
# el valor de cada cancion y el promedio del género al que corresponde
for col in audio_cols:
    df[f"{col}_minus_genre_avg"] = df[col] - df[f"genre_avg_{col}"]


## Procesando otras variables

In [15]:
# One hot encoding de time_signature
df = pd.get_dummies(
    df,
    columns=["time_signature"]
)

In [16]:
# Se elimina el ID de la canción
df = df.drop_duplicates(subset= "track_id")

In [17]:
# 
X = df.drop(columns=["popularity", 'track_id', 'track_genre', "artists", "artist_list"])
y = df["popularity"]

print(X.shape)
print(y.shape)

(89740, 40)
(89740,)


In [20]:
# Verificando que solo artist_pop_max tiene columnas nan
X.isna().mean().sort_values(ascending=False).head()

artist_pop_max    0.588077
explicit          0.000000
energy            0.000000
danceability      0.000000
mode              0.000000
dtype: float64

In [ ]:
# Pongo esta celda para que no corran el modelo por accidente 
asdfasdfasdfasfd

# Entrenando modelo

In [42]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, KFold
from xgboost import XGBRegressor


In [43]:
X.shape

(89740, 40)

In [44]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold

xgb = XGBRegressor(
    objective="reg:squarederror",
    n_jobs=-1
)

param_dist = {
    "n_estimators": [10, 100, 1000, 5000],
    "max_depth": [5, 10, 20, 30],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5]
}

In [45]:
cv = KFold(
    n_splits=5,
    shuffle=True,
)


In [46]:
search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    verbose=2,
    n_jobs=-1
)

In [47]:
search.fit(X, y)

print("Best RMSE:", -search.best_score_)
print("Best params:")
print(search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


ValueError: 
All the 250 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
250 fits failed with the following error:
Traceback (most recent call last):
  File "/home/bruno/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bruno/miniconda3/lib/python3.13/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
  File "/home/bruno/miniconda3/lib/python3.13/site-packages/xgboost/sklearn.py", line 1368, in fit
    self._Booster = train(
                    ~~~~~^
        params,
        ^^^^^^^
    ...<9 lines>...
        callbacks=self.callbacks,
        ^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File

In [ ]:
best_model = search.best_estimator_

importance = (
    pd.Series(
        best_model.feature_importances_,
        index=X.columns
    )
    .sort_values(ascending=False)
)

print(importance.head(30))

In [ ]:
plt.plot(y, best_model.predict(X), '.')
plt.plot([0,100], [0,100])

In [ ]:

plot_df = pd.DataFrame({
    "actual": y,
    "predicted": best_model.predict(X).clip(0)
})

fig = px.density_heatmap(
    plot_df,
    x="actual",
    y="predicted",
    nbinsx=100,
    nbinsy=100,
    marginal_x="histogram",
    marginal_y="histogram",
)

# Perfect prediction line
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=100,
    y1=100,
    line=dict(
        dash="dash",
        width=2
    )
)

fig.update_layout(
    title="Predicted vs Actual Popularity",
    xaxis_title="Actual Popularity",
    yaxis_title="Predicted Popularity",
    width=1000,
    height=900
)

fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

plot_df = pd.DataFrame({
    "actual": y,
    "predicted": best_model.predict(X)
})

plot_df["residual"] = (
    plot_df["actual"] - plot_df["predicted"]
)

residual_stats = (
    plot_df
    .groupby("actual")
    .agg(
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        count=("residual", "size")
    )
    .reset_index()
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=residual_stats["actual"],
        y=residual_stats["mean_residual"],
        mode="lines+markers",
        name="Mean residual"
    )
)

fig.add_trace(
    go.Scatter(
        x=residual_stats["actual"],
        y=residual_stats["median_residual"],
        mode="lines+markers",
        name="Median residual"
    )
)

fig.add_hline(
    y=0,
    line_dash="dash",
    annotation_text="Perfectly unbiased"
)

fig.update_layout(
    title="Prediction Bias by Actual Popularity",
    xaxis_title="Actual Popularity",
    yaxis_title="Actual - Predicted",
    width=1000,
    height=600
)

fig.add_trace(
    go.Bar(
        x=residual_stats["actual"],
        y=residual_stats["count"],
        name="Count",
        yaxis="y2",
        opacity=0.3
    )
)

fig.update_layout(
    yaxis2=dict(
        overlaying="y",
        side="right",
        title="Number of Songs"
    )
)
fig.show()

In [ ]:
from sklearn.model_selection import cross_val_score

r2_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print("R² per fold:", r2_scores)
print("Mean R²:", r2_scores.mean())
print("Std R²:", r2_scores.std())